In [1]:


from pyspark.sql.functions import col, current_timestamp, max as spark_max
from delta.tables import DeltaTable

bronze_table = "LH_Bronze.bulk_rides"
staging_schema = "LH_Staging"
staging_table = "LH_Staging.staging_rides"
watermark_table = "LH_Staging.load_control"
source_name = "EH_Bronze"
business_key = "ride_id"
watermark_column = "bronze_ingestion_timestamp"

kusto_uri = "https://trd-0fhzhy8rxk3smg5ka5.z5.kusto.fabric.microsoft.com"
kusto_database = "Stream_Database"
kusto_query = """
EH_Bronze
| extend bronze_ingestion_timestamp = ingestion_time()
"""


if not spark.catalog.tableExists(staging_table) or not spark.catalog.tableExists(watermark_table):
    raise RuntimeError("Staging setup is missing. Run 01_Staging_Setup first.")

StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 3, Finished, Available, Finished, False)

In [2]:

access_token = notebookutils.credentials.getToken(kusto_uri)
stream_df = (
    spark.read
    .format("com.microsoft.kusto.spark.synapse.datasource")
    .option("accessToken", access_token)
    .option("kustoCluster", kusto_uri)
    .option("kustoDatabase", kusto_database)
    .option("kustoQuery", kusto_query)
    .load()
)
print("Eventhouse records:", stream_df.count())


StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 4, Finished, Available, Finished, False)

Eventhouse records: 126


In [3]:

watermark_row = (
    spark.table(watermark_table)
    .filter(col("source_name") == source_name)
    .select("last_processed_timestamp")
    .first()
)
last_watermark = watermark_row["last_processed_timestamp"] if watermark_row else None
print("Current Eventhouse ingestion watermark:", last_watermark)


StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 5, Finished, Available, Finished, False)

Current Eventhouse ingestion watermark: 2026-10-01 17:42:45.133472


In [4]:

if last_watermark is None:
    incremental_df = stream_df
else:
    incremental_df = stream_df.filter(col(watermark_column) > last_watermark)

incremental_count = incremental_df.count()
print("Streaming records after watermark:", incremental_count)


StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 6, Finished, Available, Finished, False)

Streaming records after watermark: 0


In [5]:

existing_ride_ids = spark.table(staging_table).select(business_key).distinct()
new_streaming_df = incremental_df.join(existing_ride_ids, on=business_key, how="left_anti")
new_count = new_streaming_df.count()
print("New streaming rides to append:", new_count)


StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 7, Finished, Available, Finished, False)

New streaming rides to append: 0


In [6]:

if new_count > 0:
    new_streaming_df = new_streaming_df.withColumn("staging_load_timestamp", current_timestamp())

    target_schema = spark.table(staging_table).schema
    target_types = {f.name: f.dataType for f in target_schema.fields}

    missing_columns = [c for c in target_types if c not in new_streaming_df.columns]
    extra_columns = [c for c in new_streaming_df.columns if c not in target_types]

    if missing_columns:
        raise ValueError(f"Incremental source is missing target columns: {missing_columns}")

    aligned_df = new_streaming_df.select([
        col(c).cast(target_types[c]).alias(c) for c in target_types
    ])

    (aligned_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(staging_table))

    print(f"Appended {new_count} streaming rides.")
    if extra_columns:
        print("Ignored source-only columns:", extra_columns)
else:
    print("No new streaming rides to append.")


StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 8, Finished, Available, Finished, False)

No new streaming rides to append.


In [7]:
# 8. UPDATE WATERMARK ONLY AFTER THE LOAD STEP SUCCEEDS
# Advance to the max ingestion timestamp observed in the processed Eventhouse batch.
if incremental_count > 0:
    new_watermark = (
        incremental_df
        .agg(spark_max(watermark_column).alias("max_ts"))
        .first()["max_ts"]
    )

    watermark_update_df = spark.createDataFrame(
        [(source_name, new_watermark)],
        "source_name string, last_processed_timestamp timestamp"
    )

    control_delta = DeltaTable.forName(spark, watermark_table)
    (control_delta.alias("t")
        .merge(watermark_update_df.alias("s"), "t.source_name = s.source_name")
        .whenMatchedUpdate(set={"last_processed_timestamp": "s.last_processed_timestamp"})
        .whenNotMatchedInsert(values={
            "source_name": "s.source_name",
            "last_processed_timestamp": "s.last_processed_timestamp"
        })
        .execute())

    print("Watermark updated to:", new_watermark)
else:
    print("Watermark unchanged.")


StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 9, Finished, Available, Finished, False)

Watermark unchanged.


In [8]:
# 9. FINAL VALIDATION / SUMMARY
final_staging_count = spark.table(staging_table).count()
final_watermark_row = (
    spark.table(watermark_table)
    .filter(col("source_name") == source_name)
    .select("last_processed_timestamp")
    .first()
)
final_watermark = final_watermark_row["last_processed_timestamp"] if final_watermark_row else None

print("======================================")
print("          STAGING LOAD SUMMARY")
print("======================================")
print(f"Streaming records after watermark : {incremental_count}")
print(f"New streaming rides appended      : {new_count}")
print(f"Total staging records             : {final_staging_count}")
print(f"Current watermark                 : {final_watermark}")
print("======================================")

StatementMeta(, e0630cc0-4ae1-4088-abdb-02f81d29d1e6, 10, Finished, Available, Finished, False)

          STAGING LOAD SUMMARY
Streaming records after watermark : 0
New streaming rides appended      : 0
Total staging records             : 2126
Current watermark                 : 2026-10-01 17:42:45.133472
